# 01 — Extracción y Carga de Datos NHANES 2013-2014

**Objetivo:** Descargar los módulos clínicos y el archivo de mortalidad de NHANES directamente desde los servidores del CDC.

**Pregunta de negocio:** ¿Podemos predecir el riesgo de muerte a partir de variables clínicas y demográficas de adultos estadounidenses?

Este notebook replica la lógica de los nodos `download_*` y `load_mortality` de `nodes.py`.

> **Nota técnica:** El CDC usa un CDN (Akamai) que bloquea requests sin `User-Agent`. Agregamos un header de navegador real para que el servidor entregue el archivo binario `.XPT` en lugar de una página HTML de error.

In [1]:
%pip install pandas pyreadstat requests pyarrow --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import gzip
import io
import logging
from pathlib import Path

import pandas as pd
import pyreadstat
import requests

logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')
logger = logging.getLogger(__name__)

BASE_DIR = Path('data/01_raw/nhanes_2013_2014')
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Header que simula un navegador real — necesario porque el CDN del CDC
# bloquea requests sin User-Agent y devuelve HTML en lugar del .XPT
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    )
}

print('Directorio de salida:', BASE_DIR.resolve())
print('Headers configurados: OK')

Directorio de salida: C:\Users\nholc\Desktop\ev3_nanhes\notebooks\data\01_raw\nhanes_2013_2014
Headers configurados: OK


## 1. Función de descarga genérica para archivos XPT

Los archivos `.XPT` son el formato estándar de SAS que usa el CDC para publicar NHANES. Usamos `pyreadstat` para leerlos directamente desde un stream de bytes sin guardar el `.xpt` en disco.

Si el CDC no responde (timeout, cambio de URL), el código usa el `.parquet` local como fallback, lo que garantiza que el equipo pueda trabajar offline.

In [3]:
URL_CANDIDATES = {
    'demo': [
        'https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/DEMO_H.XPT',
        'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/DEMO_H.xpt',
    ],
    'biopro': [
        'https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/BIOPRO_H.XPT',
        'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/BIOPRO_H.xpt',
    ],
    'bmx': [
        'https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/BMX_H.XPT',
        'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/BMX_H.xpt',
    ],
    'bpx': [
        'https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/BPX_H.XPT',
        'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/BPX_H.xpt',
    ],
    'smq': [
        'https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/SMQ_H.XPT',
        'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/SMQ_H.xpt',
    ],
}

def _looks_like_xpt(content: bytes) -> bool:
    head = content[:200].upper()
    return b'HEADER RECORD' in head or b'SAS' in head

def download_xpt(name: str) -> pd.DataFrame:
    """Descarga archivo XPT desde CDC con múltiples URLs o usa fallback local."""
    urls = URL_CANDIDATES.get(name.lower())
    if not urls:
        raise ValueError(f"Dataset '{name}' no reconocido. Opciones: {list(URL_CANDIDATES.keys())}")

    last_error = None
    for url in urls:
        try:
            logger.info('Descargando %s desde %s ...', name.upper(), url)
            resp = requests.get(url, headers=HEADERS, timeout=60, allow_redirects=True)
            resp.raise_for_status()
            content = resp.content
            ctype = resp.headers.get('Content-Type', '').lower()
            if content[:2] == b'\x1f\x8b':
                content = gzip.decompress(content)
            if 'html' in ctype or (content.lstrip().startswith(b'<!DOCTYPE html') or content.lstrip().startswith(b'<html')):
                raise ValueError('CDC devolvió HTML en lugar de XPT')
            if not _looks_like_xpt(content):
                raise ValueError('La respuesta no parece un archivo XPT válido')
            df, _ = pyreadstat.read_xport(io.BytesIO(content))
            logger.info('[OK] %s descargado: %d filas, %d columnas', name.upper(), *df.shape)
            if 'SEQN' in df.columns:
                df['SEQN'] = pd.to_numeric(df['SEQN'], errors='coerce').astype('Int64')
            return df
        except Exception as e:
            last_error = e
            logger.warning('Fallo en %s: %s', url, e)

    local = BASE_DIR / f'{name.lower()}.parquet'
    if local.exists():
        logger.warning('Usando fallback local para %s', name.upper())
        df = pd.read_parquet(local)
        if 'SEQN' in df.columns:
            df['SEQN'] = pd.to_numeric(df['SEQN'], errors='coerce').astype('Int64')
        return df

    raise ConnectionError(
        f'No se pudo obtener {name.upper()} desde ninguna URL oficial. Último error: {last_error}\n'
        f'Tampoco existe archivo local en {local}'
    )


## 2. Descarga de los módulos clínicos

| Módulo | Archivo CDC | Contenido |
|---|---|---|
| DEMO | DEMO_H.XPT | Edad, sexo, etnia, ingresos |
| BIOPRO | BIOPRO_H.XPT | Laboratorio de bioquímica (creatinina, glucosa, etc.) |
| BMX | BMX_H.XPT | Medidas corporales (BMI, talla, peso) |
| BPX | BPX_H.XPT | Presión arterial |
| SMQ | SMQ_H.XPT | Historial de tabaquismo |

In [4]:
demo   = download_xpt('demo')
biopro = download_xpt('biopro')
bmx    = download_xpt('bmx')
bpx    = download_xpt('bpx')
smq    = download_xpt('smq')

[06/09/26 16:44:52] WARNING  Fallo en https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/DEMO_H.XPT: CDC    ]8;id=10493332;file://C:\Users\nholc\AppData\Local\Temp\ipykernel_17668\218513741.py\218513741.py]8;;\:]8;id=10493333;file://C:\Users\nholc\AppData\Local\Temp\ipykernel_17668\218513741.py#55\55]8;;\
                             devolvió HTML en lugar de XPT                                                         

[06/09/26 16:44:54] WARNING  Fallo en https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/BIOPRO_H.XPT: CDC  ]8;id=10493338;file://C:\Users\nholc\AppData\Local\Temp\ipykernel_17668\218513741.py\218513741.py]8;;\:]8;id=10493339;file://C:\Users\nholc\AppData\Local\Temp\ipykernel_17668\218513741.py#55\55]8;;\
                             devolvió HTML en lugar de XPT                                                         

[06/09/26 16:44:56] WARNING  Fallo en https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/BMX_H.XPT: CDC     ]8;id=10493344;file://C:\Users\nholc\AppData\Local\Temp\ipykernel_17668\218513741.py\218513741.py]8;;\:]8;id=10493345;file://C:\Users\nholc\AppData\Local\Temp\ipykernel_17668\218513741.py#55\55]8;;\
                             devolvió HTML en lugar de XPT                                                         

[06/09/26 16:44:57] WARNING  Fallo en https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/BPX_H.XPT: CDC     ]8;id=10493350;file://C:\Users\nholc\AppData\Local\Temp\ipykernel_17668\218513741.py\218513741.py]8;;\:]8;id=10493351;file://C:\Users\nholc\AppData\Local\Temp\ipykernel_17668\218513741.py#55\55]8;;\
                             devolvió HTML en lugar de XPT                                                         

[06/09/26 16:44:59] WARNING  Fallo en https://wwwn.cdc.gov/Nchs/Nhanes/2013-2014/SMQ_H.XPT: CDC     ]8;id=10493356;file://C:\Users\nholc\AppData\Local\Temp\ipykernel_17668\218513741.py\218513741.py]8;;\:]8;id=10493357;file://C:\Users\nholc\AppData\Local\Temp\ipykernel_17668\218513741.py#55\55]8;;\
                             devolvió HTML en lugar de XPT                                                         

### Validación rápida de cada módulo

In [5]:
for name, df in [('DEMO', demo), ('BIOPRO', biopro), ('BMX', bmx), ('BPX', bpx), ('SMQ', smq)]:
    nulos_seqn = df['SEQN'].isna().sum() if 'SEQN' in df.columns else 'N/A'
    print(f'{name:8s}: {df.shape[0]:>6,} filas × {df.shape[1]:>3} cols | SEQN nulos: {nulos_seqn}')
print()
demo.head(3)

DEMO    : 10,175 filas ×  47 cols | SEQN nulos: 0
BIOPRO  :  6,979 filas ×  38 cols | SEQN nulos: 0
BMX     :  9,813 filas ×  26 cols | SEQN nulos: 0
BPX     :  9,813 filas ×  23 cols | SEQN nulos: 0
SMQ     :  7,168 filas ×  32 cols | SEQN nulos: 0



,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DMDHREDU,DMDHRMAR,DMDHSEDU,WTINT2YR,WTMEC2YR,SDMVPSU,SDMVSTRA,INDHHIN2,INDFMIN2,INDFMPIR
0,73557,8.0,2.0,1.0,69.0,NaN,4.0,4.0,1.0,NaN,...,3.0,4.0,NaN,13281.237386,13481.042095,1.0,112.0,4.0,4.0,0.84
1,73558,8.0,2.0,1.0,54.0,NaN,3.0,3.0,1.0,NaN,...,3.0,1.0,1.0,23682.057386,24471.769625,1.0,108.0,7.0,7.0,1.78
2,73559,8.0,2.0,1.0,72.0,NaN,3.0,3.0,2.0,NaN,...,4.0,1.0,3.0,57214.803319,57193.285376,1.0,109.0,10.0,10.0,4.51


## 3. Archivo de mortalidad (formato .dat de ancho fijo)

El CDC vincula los datos de NHANES con el **National Death Index (NDI)**. El archivo `.dat` usa posiciones fijas de caracteres por columna (formato COBOL heredado).

Filtramos `eligstat == '1'` para quedarnos solo con participantes **elegibles para análisis de mortalidad** (adultos con seguimiento completo). Los demás (niños, participantes sin seguimiento) se excluyen.

In [6]:
MORT_URLS = [
    'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/NHANES_2013_2014_MORT_2019_PUBLIC.dat',
    'https://www.cdc.gov/nchs/data-linkage/mortality-public.htm',
]

COLS = [
    ('publicid',     ( 0, 14)),
    ('eligstat',     (14, 15)),
    ('mortstat',     (15, 16)),
    ('ucod_leading', (16, 19)),
    ('diabetes',     (19, 20)),
    ('hyperten',     (20, 21)),
    ('permth_int',   (42, 45)),
    ('permth_exm',   (45, 48)),
]

def load_mortality() -> pd.DataFrame:
    """Descarga mortalidad NHANES desde CDC o usa fallback local."""
    last_error = None
    stream = None
    for url in MORT_URLS:
        try:
            logger.info('Descargando mortalidad desde %s ...', url)
            resp = requests.get(url, headers=HEADERS, timeout=60, allow_redirects=True)
            resp.raise_for_status()
            content = resp.content
            ctype = resp.headers.get('Content-Type', '').lower()
            if content[:2] == b'\x1f\x8b':
                content = gzip.decompress(content)
            if 'html' in ctype or (content.lstrip().startswith(b'<!DOCTYPE html') or content.lstrip().startswith(b'<html')):
                raise ValueError('CDC devolvió HTML en lugar del .dat')
            stream = io.BytesIO(content)
            break
        except Exception as e:
            last_error = e
            logger.warning('Fallo en %s: %s', url, e)

    if stream is None:
        local = BASE_DIR / 'mortality.parquet'
        if local.exists():
            logger.warning('Usando fallback local para mortalidad')
            return pd.read_parquet(local)
        raise ConnectionError(
            f'No se pudo obtener mortalidad desde ninguna URL oficial. Último error: {last_error}\n'
            f'Tampoco existe archivo local en {local}'
        )

    mort = pd.read_fwf(stream, colspecs=[c[1] for c in COLS], names=[c[0] for c in COLS], dtype=str)
    mort['SEQN'] = pd.to_numeric(mort['publicid'].str.strip(), errors='coerce').astype('Int64')
    mort['eligstat'] = mort['eligstat'].str.strip()
    mort['mortstat'] = pd.to_numeric(mort['mortstat'], errors='coerce')
    mort['permth_int'] = pd.to_numeric(mort['permth_int'], errors='coerce')
    mort['permth_exm'] = pd.to_numeric(mort['permth_exm'], errors='coerce')
    mort = mort[mort['eligstat'] == '1'].dropna(subset=['SEQN', 'mortstat', 'permth_int']).copy()
    logger.info('[OK] Mortalidad: %d participantes elegibles | Tasa: %.2f%%', len(mort), mort['mortstat'].mean() * 100)
    return mort

mortality = load_mortality()


In [7]:
print(f'Mortalidad: {mortality.shape[0]:,} participantes')
print(f'Tasa observada de muerte: {mortality["mortstat"].mean()*100:.2f}%')
mortality[['SEQN', 'eligstat', 'mortstat', 'permth_int', 'ucod_leading']].head(5)

Mortalidad: 6,100 participantes
Tasa observada de muerte: 7.66%


,SEQN,eligstat,mortstat,permth_int,ucod_leading
0,73557,1,0.0,78.0,NaN
1,73558,1,1.0,58.0,007
2,73559,1,0.0,83.0,NaN
4,73561,1,1.0,10.0,010
5,73562,1,0.0,68.0,NaN


## 4. Persistencia en disco (replicando el catálogo Kedro)

Guardamos cada módulo como `.parquet` en las rutas definidas en `catalog.yml`. En producción Kedro hace esto automáticamente; aquí lo hacemos explícito para entender el flujo.

In [8]:
datasets = {
    'demo': demo, 'biopro': biopro, 'bmx': bmx,
    'bpx': bpx, 'smq': smq, 'mortality': mortality,
}
for name, df in datasets.items():
    path = BASE_DIR / f'{name}.parquet'
    df.to_parquet(path, index=False)
    print(f'✓ Guardado: {path}  ({len(df):,} filas)')
print('\n✓ Todos los raw datasets guardados correctamente.')

✓ Guardado: data\01_raw\nhanes_2013_2014\demo.parquet  (10,175 filas)
✓ Guardado: data\01_raw\nhanes_2013_2014\biopro.parquet  (6,979 filas)
✓ Guardado: data\01_raw\nhanes_2013_2014\bmx.parquet  (9,813 filas)
✓ Guardado: data\01_raw\nhanes_2013_2014\bpx.parquet  (9,813 filas)
✓ Guardado: data\01_raw\nhanes_2013_2014\smq.parquet  (7,168 filas)
✓ Guardado: data\01_raw\nhanes_2013_2014\mortality.parquet  (6,100 filas)

✓ Todos los raw datasets guardados correctamente.
